# Assignment 04 — Novel CNN Architectures for Lightweight Deep Learning

## Title
**Implementation of Efficient CNN Models: MobileNetV2, DenseNet121, and EfficientNetB0**

## Objective
Implement modern CNN architectures and evaluate their suitability for **resource-constrained systems**.

## Dataset
We use **CIFAR-10** (10 classes). (You can switch to MNIST/Fashion-MNIST later if required.)

## Deliverables (inside this notebook)
- ✅ Source code (TensorFlow/Keras)
- ✅ Comparison table (accuracy, params, size, inference speed, memory)
- ✅ Performance graphs
- ✅ CPU-only test for one model
- ✅ Final recommendation report
- ✅ All artifacts saved to Google Drive (`OUTPUT_DIR`)

---

### Colab runtime
`Runtime → Change runtime type → GPU (T4)`


In [ ]:
# =============================
# 1) Setup
# =============================
import os
import time
import json
import math
import random
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'


## 1.1) Google Drive (Save all outputs)
This mounts Google Drive and creates an output folder for **models, plots, tables, and reports**.


In [ ]:
# Mount Drive (Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception as e:
    IN_COLAB = False
    print('Not running in Colab (Drive mount skipped):', e)

DRIVE_ROOT = '/content/drive/MyDrive'
PROJECT_DIR = f"{DRIVE_ROOT}/COMP-443/Assignment_04_Efficient_CNNs"
LOCAL_DIR = '/content/assignment04_outputs'

OUTPUT_DIR = PROJECT_DIR if IN_COLAB else LOCAL_DIR
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('OUTPUT_DIR =', OUTPUT_DIR)


## 2) Dataset: CIFAR-10
All three backbones will use the **same input size** and **same training settings** for a fair comparison.

- Input size: **224×224** (common for MobileNetV2, DenseNet121, EfficientNetB0)
- Augmentation: flip + small contrast
- Training: head training + optional light fine-tune


In [ ]:
# =============================
# 2) Load CIFAR-10
# =============================
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

y_train = y_train.squeeze().astype('int32')
y_test  = y_test.squeeze().astype('int32')

class_names = [
    'airplane','automobile','bird','cat','deer',
    'dog','frog','horse','ship','truck'
]
num_classes = len(class_names)
print('Train:', x_train.shape, y_train.shape)
print('Test :', x_test.shape, y_test.shape)


In [ ]:
# Train/val split

def make_splits(x, y, val_fraction=0.1):
    n = x.shape[0]
    n_val = int(n * val_fraction)
    idx = np.arange(n)
    rng = np.random.default_rng(SEED)
    rng.shuffle(idx)
    val_idx = idx[:n_val]
    tr_idx  = idx[n_val:]
    return (x[tr_idx], y[tr_idx]), (x[val_idx], y[val_idx])

(x_tr, y_tr), (x_val, y_val) = make_splits(x_train, y_train, 0.1)
print('Train split:', x_tr.shape, y_tr.shape)
print('Val split  :', x_val.shape, y_val.shape)


In [ ]:
# =============================
# 3) tf.data pipeline
# =============================
AUTOTUNE = tf.data.AUTOTUNE
BATCH_SIZE = 64
IMAGE_SIZE = (224, 224)

augment = keras.Sequential([
    layers.RandomFlip('horizontal', seed=SEED),
    layers.RandomContrast(0.1, seed=SEED),
], name='augmentation')

# Preprocess functions (model-specific)
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_preprocess


def make_dataset(x, y, batch_size, image_size, preprocess_fn, training):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(10_000, seed=SEED, reshuffle_each_iteration=True)

    def _map(img, label):
        img = tf.cast(img, tf.float32)
        if training:
            img = augment(img)
        img = tf.image.resize(img, image_size, method='bilinear')
        img = preprocess_fn(img)
        return img, tf.one_hot(label, num_classes)

    ds = ds.map(_map, num_parallel_calls=AUTOTUNE)
    # Cache helps performance in Colab RAM
    ds = ds.cache()
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds


def make_all_datasets(preprocess_fn):
    train_ds = make_dataset(x_tr, y_tr, BATCH_SIZE, IMAGE_SIZE, preprocess_fn, training=True)
    val_ds   = make_dataset(x_val, y_val, BATCH_SIZE, IMAGE_SIZE, preprocess_fn, training=False)
    test_ds  = make_dataset(x_test, y_test, BATCH_SIZE, IMAGE_SIZE, preprocess_fn, training=False)
    return train_ds, val_ds, test_ds

# quick sanity check
train_ds_tmp, _, _ = make_all_datasets(mobilenet_preprocess)
xb, yb = next(iter(train_ds_tmp))
print('Batch:', xb.shape, yb.shape)


## 4) Model builders (Transfer Learning)
We use ImageNet-pretrained backbones with `include_top=False` and replace the classifier for 10 classes.


In [ ]:
# =============================
# 4) Model builders
# =============================
from tensorflow.keras.applications import MobileNetV2, DenseNet121, EfficientNetB0


def build_transfer_model(backbone_name, num_classes):
    if backbone_name == 'MobileNetV2':
        base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3), pooling='avg')
    elif backbone_name == 'DenseNet121':
        base = DenseNet121(weights='imagenet', include_top=False, input_shape=(224,224,3), pooling='avg')
    elif backbone_name == 'EfficientNetB0':
        base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3), pooling='avg')
    else:
        raise ValueError('Unknown backbone: ' + backbone_name)

    base.trainable = False

    inputs = keras.Input(shape=(224,224,3))
    x = base(inputs, training=False)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name=f'{backbone_name}_transfer')
    return model


def compile_model(model, lr):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )


def fit_with_timing(model, train_ds, val_ds, epochs, callbacks=None):
    t0 = time.perf_counter()
    hist = model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=callbacks or [], verbose=1)
    t1 = time.perf_counter()
    return hist, (t1 - t0)


## 5) Train all models with the same settings
We train in two stages:
1) Train classifier head (backbone frozen)
2) Light fine-tuning (unfreeze last N layers) with lower LR

You can reduce epochs if you want faster execution.


In [ ]:
# =============================
# 5) Training config
# =============================
EPOCHS_HEAD = 8
EPOCHS_FT = 7
DO_FINE_TUNE = True

callbacks = [
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1),
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
]


In [ ]:
# Helper: fine-tune last N layers

def fine_tune(model, backbone_substring, unfreeze_last_n):
    # Locate backbone
    backbone = None
    for layer in model.layers:
        if backbone_substring.lower() in layer.name.lower():
            backbone = layer
            break
    if backbone is None:
        # fallback: first layer that is a keras.Model
        for layer in model.layers:
            if isinstance(layer, keras.Model):
                backbone = layer
                break

    backbone.trainable = True
    layers_list = backbone.layers
    for l in layers_list[:-unfreeze_last_n]:
        l.trainable = False

    return backbone


In [ ]:
# =============================
# 5.1) MobileNetV2
# =============================
train_mnet, val_mnet, test_mnet = make_all_datasets(mobilenet_preprocess)

mobilenet_model = build_transfer_model('MobileNetV2', num_classes)
compile_model(mobilenet_model, lr=1e-3)

mnet_hist_head, mnet_time_head = fit_with_timing(mobilenet_model, train_mnet, val_mnet, EPOCHS_HEAD, callbacks)

mnet_time_ft = 0.0
mnet_hist_ft = None
if DO_FINE_TUNE:
    fine_tune(mobilenet_model, 'mobilenet', unfreeze_last_n=40)
    compile_model(mobilenet_model, lr=1e-4)
    mnet_hist_ft, mnet_time_ft = fit_with_timing(mobilenet_model, train_mnet, val_mnet, EPOCHS_FT, callbacks)

print('MobileNetV2 time (min):', (mnet_time_head+mnet_time_ft)/60)


In [ ]:
# =============================
# 5.2) DenseNet121
# =============================
train_dnet, val_dnet, test_dnet = make_all_datasets(densenet_preprocess)

densenet_model = build_transfer_model('DenseNet121', num_classes)
compile_model(densenet_model, lr=1e-3)

dnet_hist_head, dnet_time_head = fit_with_timing(densenet_model, train_dnet, val_dnet, EPOCHS_HEAD, callbacks)

dnet_time_ft = 0.0
dnet_hist_ft = None
if DO_FINE_TUNE:
    fine_tune(densenet_model, 'densenet', unfreeze_last_n=60)
    compile_model(densenet_model, lr=1e-4)
    dnet_hist_ft, dnet_time_ft = fit_with_timing(densenet_model, train_dnet, val_dnet, EPOCHS_FT, callbacks)

print('DenseNet121 time (min):', (dnet_time_head+dnet_time_ft)/60)


In [ ]:
# =============================
# 5.3) EfficientNetB0
# =============================
train_enet, val_enet, test_enet = make_all_datasets(effnet_preprocess)

effnet_model = build_transfer_model('EfficientNetB0', num_classes)
compile_model(effnet_model, lr=1e-3)

enet_hist_head, enet_time_head = fit_with_timing(effnet_model, train_enet, val_enet, EPOCHS_HEAD, callbacks)

enet_time_ft = 0.0
enet_hist_ft = None
if DO_FINE_TUNE:
    fine_tune(effnet_model, 'efficientnet', unfreeze_last_n=50)
    compile_model(effnet_model, lr=1e-4)
    enet_hist_ft, enet_time_ft = fit_with_timing(effnet_model, train_enet, val_enet, EPOCHS_FT, callbacks)

print('EfficientNetB0 time (min):', (enet_time_head+enet_time_ft)/60)


## 6) Evaluation: Accuracy, Parameters, Model Size
We evaluate test accuracy/loss and save each model to Drive to measure file size.


In [ ]:
# =============================
# 6) Evaluate & save
# =============================

def eval_and_save(model, test_ds, name):
    loss, acc = model.evaluate(test_ds, verbose=0)

    model_path = os.path.join(OUTPUT_DIR, f'{name}.keras')
    model.save(model_path, include_optimizer=False)
    size_mb = os.path.getsize(model_path) / (1024**2)

    return {
        'model': name,
        'test_accuracy': float(acc),
        'test_loss': float(loss),
        'params': int(model.count_params()),
        'model_size_mb': float(size_mb),
        'model_path': model_path,
    }

mnet_metrics = eval_and_save(mobilenet_model, test_mnet, 'mobilenetv2_cifar10_transfer')
dnet_metrics = eval_and_save(densenet_model, test_dnet, 'densenet121_cifar10_transfer')
enet_metrics = eval_and_save(effnet_model, test_enet, 'efficientnetb0_cifar10_transfer')

mnet_metrics, dnet_metrics, enet_metrics


## 7) Inference Speed Benchmark (GPU + CPU)
We benchmark **average inference time per batch**.

Notes:
- We run a warmup first.
- We measure on GPU if available.
- We also measure CPU inference for **one model** as required.


In [ ]:
# =============================
# 7) Speed + memory benchmark
# =============================

def benchmark_inference(model, sample_ds, device=None, steps=50, warmup=10):
    it = iter(sample_ds)

    # compile predict to graph for stable timing
    @tf.function
    def _predict(x):
        return model(x, training=False)

    # warmup
    for _ in range(warmup):
        x, _ = next(it)
        if device:
            with tf.device(device):
                _ = _predict(x)
        else:
            _ = _predict(x)

    # timed
    times = []
    for _ in range(steps):
        x, _ = next(it)
        t0 = time.perf_counter()
        if device:
            with tf.device(device):
                _ = _predict(x)
        else:
            _ = _predict(x)
        # ensure sync
        _ = tf.reduce_sum(_)
        t1 = time.perf_counter()
        times.append(t1 - t0)

    return float(np.mean(times)), float(np.std(times))


def get_rss_mb():
    try:
        import psutil
        import os as _os
        p = psutil.Process(_os.getpid())
        return p.memory_info().rss / (1024**2)
    except Exception:
        return None

# GPU benchmark (if available)
gpu_device = '/GPU:0' if tf.config.list_physical_devices('GPU') else None

rss_before = get_rss_mb()

mnet_gpu_mean, mnet_gpu_std = benchmark_inference(mobilenet_model, test_mnet, device=gpu_device)
dnet_gpu_mean, dnet_gpu_std = benchmark_inference(densenet_model, test_dnet, device=gpu_device)
enet_gpu_mean, enet_gpu_std = benchmark_inference(effnet_model, test_enet, device=gpu_device)

rss_after = get_rss_mb()

print('GPU mean seconds/batch:')
print('MobileNetV2   ', mnet_gpu_mean)
print('DenseNet121   ', dnet_gpu_mean)
print('EfficientNetB0', enet_gpu_mean)
print('RSS before/after (MB):', rss_before, rss_after)


In [ ]:
# CPU-only test (required): run MobileNetV2 on CPU
mnet_cpu_mean, mnet_cpu_std = benchmark_inference(mobilenet_model, test_mnet, device='/CPU:0')
print('MobileNetV2 CPU mean seconds/batch:', mnet_cpu_mean)


## 8) Training Curves + Comparison Graphs


In [ ]:
# =============================
# 8) Curves
# =============================

def merge_histories(h1, h2):
    if h2 is None:
        return h1.history
    merged = {k: list(h1.history.get(k, [])) + list(h2.history.get(k, [])) for k in set(h1.history) | set(h2.history)}
    return merged

mnet_hist = merge_histories(mnet_hist_head, mnet_hist_ft)
dnet_hist = merge_histories(dnet_hist_head, dnet_hist_ft)
enet_hist = merge_histories(enet_hist_head, enet_hist_ft)


def plot_curves(hist, title, fname):
    acc = hist.get('accuracy', [])
    val_acc = hist.get('val_accuracy', [])
    loss = hist.get('loss', [])
    val_loss = hist.get('val_loss', [])
    epochs = range(1, len(acc)+1)

    plt.figure(figsize=(12,4))

    plt.subplot(1,2,1)
    plt.plot(epochs, acc, label='Train')
    plt.plot(epochs, val_acc, label='Val')
    plt.title(f'{title} Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(1,2,2)
    plt.plot(epochs, loss, label='Train')
    plt.plot(epochs, val_loss, label='Val')
    plt.title(f'{title} Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    out = os.path.join(OUTPUT_DIR, fname)
    plt.savefig(out, dpi=200, bbox_inches='tight')
    plt.show()
    print('Saved', out)

plot_curves(mnet_hist, 'MobileNetV2', 'mobilenetv2_curves.png')
plot_curves(dnet_hist, 'DenseNet121', 'densenet121_curves.png')
plot_curves(enet_hist, 'EfficientNetB0', 'efficientnetb0_curves.png')


In [ ]:
# =============================
# 9) Comparison table + graphs
# =============================
import pandas as pd

results = [
    {
        **mnet_metrics,
        'train_time_min': float((mnet_time_head+mnet_time_ft)/60),
        'gpu_sec_per_batch': float(mnet_gpu_mean),
        'cpu_sec_per_batch': float(mnet_cpu_mean),
    },
    {
        **dnet_metrics,
        'train_time_min': float((dnet_time_head+dnet_time_ft)/60),
        'gpu_sec_per_batch': float(dnet_gpu_mean),
        'cpu_sec_per_batch': None,
    },
    {
        **enet_metrics,
        'train_time_min': float((enet_time_head+enet_time_ft)/60),
        'gpu_sec_per_batch': float(enet_gpu_mean),
        'cpu_sec_per_batch': None,
    },
]

df = pd.DataFrame(results)
df


In [ ]:
# Save table
csv_path = os.path.join(OUTPUT_DIR, 'comparison_table.csv')
df.to_csv(csv_path, index=False)
print('Saved', csv_path)


In [ ]:
# Bar charts

def barplot(x, y, title, ylabel, fname):
    plt.figure(figsize=(7,4))
    plt.bar(x, y)
    plt.title(title)
    plt.ylabel(ylabel)
    plt.tight_layout()
    out = os.path.join(OUTPUT_DIR, fname)
    plt.savefig(out, dpi=200, bbox_inches='tight')
    plt.show()
    print('Saved', out)

barplot(df['model'], df['test_accuracy'], 'Test Accuracy', 'Accuracy', 'bar_accuracy.png')
barplot(df['model'], df['params'], 'Parameters', 'Count', 'bar_params.png')
barplot(df['model'], df['gpu_sec_per_batch'], 'Inference Speed (GPU)', 'Seconds / Batch', 'bar_gpu_speed.png')
barplot(df['model'], df['model_size_mb'], 'Model Size', 'MB', 'bar_model_size.png')


## 10) Final Recommendation Report (HTML + Markdown)
We generate a standalone HTML report and a Markdown report saved in `OUTPUT_DIR`.

You should edit the recommendation section based on your measured results.


In [ ]:
# =============================
# 10) Export report.html + report.md
# =============================
import datetime

report_html = f'''<!doctype html>
<html>
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Assignment 04 — Efficient CNNs Report</title>
  <style>
    body {{ font-family: Arial, sans-serif; margin: 24px; line-height: 1.45; }}
    h1,h2,h3 {{ margin-bottom: 8px; }}
    .meta {{ color: #444; margin-bottom: 18px; }}
    table {{ border-collapse: collapse; width: 100%; }}
    th, td {{ border: 1px solid #ddd; padding: 8px; text-align: left; }}
    th {{ background: #f5f5f5; }}
    figure {{ margin: 16px 0; }}
    figcaption {{ color: #555; font-size: 0.95em; margin-top: 6px; }}
    code {{ background:#f6f6f6; padding:2px 4px; }}
  </style>
</head>
<body>
  <h1>Assignment 04 Report — Efficient CNN Architectures</h1>
  <div class="meta">
    Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}<br>
    Dataset: CIFAR-10<br>
    Models: MobileNetV2, DenseNet121, EfficientNetB0
  </div>

  <h2>Comparison Table</h2>
  {df.to_html(index=False)}

  <h2>Graphs</h2>
  <ul>
    <li>Accuracy bar: <code>bar_accuracy.png</code></li>
    <li>Params bar: <code>bar_params.png</code></li>
    <li>GPU speed bar: <code>bar_gpu_speed.png</code></li>
    <li>Model size bar: <code>bar_model_size.png</code></li>
    <li>Training curves: <code>mobilenetv2_curves.png</code>, <code>densenet121_curves.png</code>, <code>efficientnetb0_curves.png</code></li>
  </ul>

  <h2>CPU-only test</h2>
  <p>MobileNetV2 CPU seconds/batch: <b>{mnet_cpu_mean:.6f}</b></p>

  <h2>Recommendations (template)</h2>
  <h3>Mobile phone</h3>
  <p><b>MobileNetV2</b> is typically a strong choice due to low parameter count and fast inference.</p>

  <h3>Cloud server</h3>
  <p><b>DenseNet121</b> or <b>EfficientNetB0</b> can be preferred if they achieve higher accuracy and inference cost is acceptable.</p>

  <h3>Real-time system</h3>
  <p>Prefer the model with the best speed/accuracy tradeoff (often <b>MobileNetV2</b> or <b>EfficientNetB0</b>).</p>

  <h2>Outputs</h2>
  <p>All artifacts saved to: <code>{OUTPUT_DIR}</code></p>
</body>
</html>
'''

html_path = os.path.join(OUTPUT_DIR, 'report.html')
with open(html_path, 'w', encoding='utf-8') as f:
    f.write(report_html)
print('Saved', html_path)

md_path = os.path.join(OUTPUT_DIR, 'report.md')
with open(md_path, 'w', encoding='utf-8') as f:
    f.write('# Assignment 04 Report — Efficient CNN Architectures
')
    f.write(f"
Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
")
    f.write(f"
Outputs folder: `{OUTPUT_DIR}`
")
    f.write('
## Models
- MobileNetV2
- DenseNet121
- EfficientNetB0
')
    f.write('
## Key files
')
    f.write('- `comparison_table.csv`
- `report.html`
- `bar_accuracy.png` `bar_params.png` `bar_gpu_speed.png` `bar_model_size.png`
')
    f.write('- `mobilenetv2_curves.png` `densenet121_curves.png` `efficientnetb0_curves.png`
')
    f.write('
## Recommendations (fill using your results)
- Mobile phone: ...
- Cloud server: ...
- Real-time system: ...
')
print('Saved', md_path)


## Bonus (Optional): Deployment
If you want, we can deploy the best model with **Streamlit** or **Gradio** in a separate notebook/script.
